# BERT によるテキスト分類

### 学習目標
1. KerasHub から事前学習済み BERT モデルを読み込む方法を学ぶ
2. 分類器と組み合わせて独自のモデルを構築する方法を学ぶ
3. ファインチューニングによって LLM（BERT）モデルを訓練する方法を学ぶ
4. 訓練済みモデルを保存して使用する方法を学ぶ
5. テキスト分類モデルを評価する方法を学ぶ

このラボでは、BERT をファインチューニングして、プレーンテキストの IMDB 映画レビューデータセットに対して感情分析（センチメント分析）を実行する方法を学びます。
モデルの訓練に加えて、テキストを適切な形式に前処理する方法も学びます。

### 始める前に

訓練に時間がかかりすぎないように、Notebook インスタンスに GPU（1 x NVIDIA Tesla T4 で十分です）がアタッチされていることを確認してください。

## BERT について

[BERT](https://arxiv.org/abs/1810.04805) やその他の Transformer エンコーダアーキテクチャは、NLP（自然言語処理）のさまざまなタスクで大きな成功を収めています。これらは、ディープラーニングモデルでの使用に適した自然言語のベクトル空間表現を計算します。BERT ファミリのモデルは、Transformer エンコーダアーキテクチャを使用して、前後のすべてのトークンの完全なコンテキスト内で入力テキストの各トークンを処理します。これが Bidirectional Encoder Representations from Transformers（Transformers による双方向エンコーダ表現）という名称の由来です。

BERT モデルは通常、大規模なテキストコーパスで事前学習され、その後特定のタスクに合わせてファインチューニングされます。

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"



In [ ]:
import datetime
import shutil

import keras
import keras_hub as hub
import matplotlib.pyplot as plt
import tensorflow as tf
from google.cloud import aiplatform

tf.get_logger().setLevel("ERROR")

GPU がアタッチされているか確認するには、以下を実行します。

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

### 感情分析

このノートブックでは、レビューのテキストに基づいて、映画レビューを「ポジティブ（*positive*）」または「ネガティブ（*negative*）」に分類する感情分析モデルを訓練します。

[Internet Movie Database](https://www.imdb.com/) からの 50,000 件の映画レビューのテキストを含む [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) を使用します。

### IMDB データセットのダウンロード

データセットをダウンロードして展開し、ディレクトリ構造を確認してみましょう。

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

path = "./bert_dataset/"
os.makedirs(path, exist_ok=True) 

dataset = keras.utils.get_file(
    "aclImdb_v1.tar.gz", url, untar=True, cache_dir=path, cache_subdir=""
)

dataset_dir = os.path.join(
    os.path.dirname(dataset), "aclImdb_v1_extracted/aclImdb"
)

train_dir = os.path.join(dataset_dir, "train")

# データの読み込みを容易にするため、未使用のフォルダを削除
remove_dir = os.path.join(train_dir, "unsup")
shutil.rmtree(remove_dir)

次に、`text_dataset_from_directory` ユーティリティを使用して、ラベル付きの `tf.data.Dataset` を作成します。

IMDB データセットはすでに訓練用とテスト用に分割されていますが、検証セットがありません。以下の `validation_split` 引数を使用して、訓練データを 80:20 に分割して検証セットを作成しましょう。

注: `validation_split` と `subset` 引数を使用する場合は、検証用と訓練用の分割が重複しないように、ランダムシードを指定するか、`shuffle=False` を渡すようにしてください。

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32
seed = 42

raw_train_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed,
)

class_names = raw_train_ds.class_names
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)

val_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed,
)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

test_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/test", batch_size=batch_size
)

test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

いくつかのレビューを見てみましょう。

In [ ]:
for text_batch, label_batch in train_ds.take(1):
    for i in range(3):
        print(f"Review: {text_batch.numpy()[i]}")
        label = label_batch.numpy()[i]
        print(f"Label : {label} ({class_names[label]})")

## Keras Hub からのモデルの読み込み

このラボでは、Small BERT と呼ばれるモデルを読み込みます。Small BERT は元の BERT と同じ一般的なアーキテクチャを持っていますが、Transformer ブロックの数が少ない、またはサイズが小さくなっています。

その他の一般的な BERT モデルには、BERT Base、ALBERT、BERT Experts、Electra などがあります。詳細については、このラボの最後にある「継続的な学習」セクションを参照してください。

後述の利用可能なモデルの他に、より大規模でさらに高い精度が得られるモデルの[複数のバージョン](https://keras.io/keras_hub/presets/)もありますが、単一の GPU でファインチューニングするには大きすぎます。

以下のコードにあるように、TF Hub の SavedModel にすべての違いがカプセル化されているため、tfhub.dev の URL を切り替えるだけでこれらの任意のモデルを試すことができます。

### ファインチューニングする BERT モデルの選択

In [ ]:
bert_encoder = hub.models.BertBackbone.from_preset(
    "bert_small_en_uncased",
    load_weights=True,
)

print(
    "*******************************\nPre-Trained Model Architecture\n*******************************"
)
bert_encoder.summary()

## 前処理モデル

テキスト入力は、BERT に入力する前に数値のトークン ID に変換され、いくつかの Tensor に配置される必要があります。KerasHub は、上記で説明した各 BERT モデルに対応する前処理モデルを提供しており、この変換を実装しています。テキストの前処理を行うために、Keras モデルの外部で純粋な Python コードを実行する必要はありません。

前処理モデルは、後でモデル内でファインチューニングするモデルと同じものである必要があります。

**TODO:** [hub.model.BertPreprocessor](https://keras.io/keras_hub/api/models/bert/bert_text_classifier_preprocessor/) を使用して前処理モデルを読み込み、ファインチューニングされたモデルを構成します。

In [ ]:
bert_preprocess_model = #TODO: ここで前処理モデルを呼び出します

前処理モデルをいくつかのテキストで試してみて、出力を確認してみましょう。

前処理モデル関数を呼び出し、text_test を渡します

In [ ]:
text_test = ["this is such an amazing movie!"]
text_preprocessed = bert_preprocessor(text_test)

# この出力ボックスは前処理された辞書内のキーを確認するのに役立ちます
print(f"Keys       : {list(text_preprocessed.keys())}")

# 1. token_ids はトークン化された文内の単語の ID です
print(f'Shape      : {text_preprocessed["token_ids"].shape}')
print(f'Token Ids   : {text_preprocessed["token_ids"][0, :12]}')

# 2. padding_mask はマスクするトークンです（マスク言語モデル）
print(f'Padding Mask : {text_preprocessed["padding_mask"][0, :12]}')

# 3. segment_ids は入力文の文 ID です。
print(f'Type Ids   : {text_preprocessed["segment_ids"][0, :12]}')

ご覧のとおり、BERT モデルが使用する前処理からの 3 つの出力（`token_ids`、`padding_mask`、`type_ids`）が得られました。

その他の重要なポイント:
- 入力は 512 トークンに切り詰められます。
- 単一文の入力であるため、`type_ids` には 1 つの値（0）しかありません。複数文の入力の場合は、各入力に対して 1 つの数値が割り当てられます。

テキストプリプロセッサは `keras.layers.Layer` オブジェクトのサブクラスです。これは、個別に前処理を行う代わりに、モデルコード内のレイヤとして組み込むことができることを意味します。

### BERT モデルの使用

BERT を独自のモデルに組み込む前に、その出力を確認してみましょう。

In [ ]:
bert_encoder = hub.models.BertBackbone.from_preset(
    "bert_small_en_uncased", load_weights=True, name="bert_encoder"
)
bert_encoder.trainable = True

In [ ]:
bert_results = bert_encoder(text_preprocessed)

print(f'Pooled Outputs Shape:{bert_results["pooled_output"].shape}')
print(f'Pooled Outputs Values:{bert_results["pooled_output"][0, :12]}')
print(f'Sequence Outputs Shape:{bert_results["sequence_output"].shape}')
print(f'Sequence Outputs Values:{bert_results["sequence_output"][0, :12]}')

BERT モデルは、`pooled_output`、`sequence_output`、`encoder_outputs` という 3 つの重要なキーを持つマップを返します。

- `pooled_output`: 各入力シーケンス全体を表します。形状は `[batch_size, H]` です。これは映画レビュー全体に対する埋め込みと考えることができます。
- `sequence_output`: コンテキスト内の各入力トークンを表します。形状は `[batch_size, seq_length, H]` です。これは映画レビュー内のすべてのトークンに対するコンテキスト埋め込みと考えることができます。

ファインチューニングには、`pooled_output` 配列を使用します。

### モデルの定義

前処理モデル、選択した BERT モデル、1 つの Dense レイヤ、および Dropout レイヤを使用して、非常にシンプルなファインチューニングモデルを作成します。

注: ベースモデルの入出力に関する詳細情報は、モデルの URL をコピーしてドキュメントページを開くことで確認できます。

**TODO:** ここで分類モデルを定義します。モデル内のレイヤの順序は次のようになります。
1. 入力層（Input Layer）
2. 前処理層（Pre-processing Layer）
3. エンコーダ層（Encoder Layer）
4. BERT 出力マップから pooled_output を使用
5. ドロップアウト層（Dropout layer）
6. sigmoid 活性化関数を持つ Dense 層

In [ ]:
def build_classifier_model(dropout_rate=0.1):
    text_input = #TODO: 1. 生テキストの入力層を定義
    
    preprocessor = #TODO: 2. 入力テキストにプリプロセッサを適用
    # 'preprocessor' 変数には次の出力辞書が保持されます:
    # {'token_ids', 'segment_ids', 'padding_mask'}
    
    encoder_outputs = #TODO: 3. BERT バックボーン / エンコーダを読み込む

    net = #TODO: 4. 文レベルの分類に 'pooled_output' を使用
    
    net = #TODO: 5. ドロップアウトレイヤを追加
    net = #TODO: 6. 分類用の Dense レイヤを追加
    return keras.Model(text_input, net)


# 前処理モデルの出力でモデルが実行できることを確認しましょう。
dropout_rate = 0.15
classifier_model = build_classifier_model(dropout_rate)
bert_raw_result = classifier_model(keras.ops.convert_to_tensor(text_test))
print(bert_raw_result)

モデルはまだ訓練されていないため、もちろん出力には意味がありません。

モデルの構造を見てみましょう。

In [ ]:
keras.utils.plot_model(classifier_model, show_layer_names=True, dpi=100)

## モデルの訓練

前処理モジュール、BERT エンコーダ、データ、分類器など、モデルを訓練するためのすべてのパーツが揃いました。

### 損失関数

これは二値分類問題であり、モデルは確率を出力するため（単一ユニットのレイヤ）、`losses.BinaryCrossentropy` 損失関数を使用します。

**TODO:** ここで損失関数と評価指標を定義します。二値分類であるため、BinaryCrossentropy と BinaryAccuracy を使用してください。

In [ ]:
loss = #TODO: 損失として BinaryCrossEntropy を定義
metrics = #TODO: 評価指標として BinaryAccuracy を定義

### オプティマイザ

ファインチューニングには、BERT が元々訓練されていたのと同じオプティマイザである「Adaptive Moments」（Adam）を使用しましょう。このオプティマイザは予測損失を最小限に抑え、（モーメントを使用せず）重み減衰（weight decay）による正則化を行います。これは [AdamW](https://keras.io/api/optimizers/adamw/) としても知られています。AdamW 最適化は、1 次モーメントと 2 次モーメントの適応的推定に基づき、重み減衰の手法を追加した確率的勾配降下法です。

過去のラボでは、一般的な選択肢である Adam オプティマイザを使用してきました。しかし、このラボでは、重み減衰を使用して汎化性能を向上させることを目的とした Adam のバリアントである AdamW を使用します。AdamW の直感的な理解とアルゴリズムについては、こちらの[論文](https://arxiv.org/abs/1711.05101)を参照してください。

学習率（`learning_rate`）については、BERT の論文に合わせて、ファインチューニング用に初期学習率を小さめに設定します（5e-5、3e-5、2e-5 の中から最適なものを選択）。

In [ ]:
epochs = 5

optimizer = keras.optimizers.AdamW(learning_rate=3e-5, weight_decay=0.004)

### BERT モデルの読み込みと訓練

先ほど作成した `classifier_model` を使用して、損失、評価指標、オプティマイザを指定してモデルをコンパイルできます。

上記で定義したオプティマイザ、損失、評価指標を使用してモデルをコンパイルします。

*注: GPU との非互換性のため、ここでは `jit_compile` という属性を `False` に設定しています。*

In [ ]:
classifier_model.compile(
    optimizer=optimizer, loss=loss, metrics=[metrics], jit_compile=False
)

注: 訓練時間は、選択した BERT モデルの複雑さによって異なります。

**TODO:** `model.fit()` を使用して数エポックの間モデルを訓練しましょう。NVIDIA T4 GPU を 1 基使用した場合、2 エポックの訓練には約 15〜20 分かかることに注意してください。

In [ ]:
print(f"Training model with {tfhub_handle_encoder}")
history = classifier_model.fit(
    # TODO: ここにモデルを訓練するための引数を追加します
)

### モデルの評価

モデルのパフォーマンスを確認してみましょう。損失（誤差を表す数値、小さいほど優れている）と精度の 2 つの値が返されます。

In [ ]:
loss, accuracy = classifier_model.evaluate(test_ds)

print(f"Loss: {loss}")
print(f"Accuracy: {accuracy}")

### 精度と損失の推移のプロット

`model.fit()` から返された `History` オブジェクトに基づいて、訓練損失と検証損失、および訓練精度と検証精度を比較のためにプロットできます。

In [ ]:
history_dict = history.history
print(history_dict.keys())

acc = history_dict["binary_accuracy"]
val_acc = history_dict["val_binary_accuracy"]
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]

epochs = range(1, len(acc) + 1)
fig = plt.figure(figsize=(10, 6))
fig.tight_layout()

plt.subplot(2, 1, 1)
# "bo" は「青の点」を表します
plt.plot(epochs, loss, "r", label="Training loss")
# "b" は「青の実線」を表します
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
# plt.xlabel('Epochs')
plt.ylabel("Loss")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(epochs, acc, "r", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")

このプロットでは、赤い線が訓練の損失と精度を表し、青い線が検証の損失と精度を表します。

### 推論用へのエクスポート

ファインチューニングしたモデルを後で使用できるように保存します。

推論用にモデルを保存するには、SAVEDMODEL_PATH で指定されたパスにエクスポートします。Keras3 はモデルを保存するためのメソッドを提供しており、このノートブックでは [model.export()](https://keras.io/api/models/model_saving_apis/export/) を使用します。この関数が選択されたのは、サービング用に特別に設計され、Google Cloud の Agent Platform へのデプロイと互換性のある TensorFlow SavedModel 形式でモデルを生成するためです。

In [ ]:
dataset_name = "imdb"
saved_model_path = "./{}_bert".format(dataset_name.replace("/", "_"))
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(saved_model_path, TIMESTAMP)


SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")

classifier_model.export(SAVEDMODEL_PATH)

メモリ内にまだあるモデルと並べて試すことができるように、モデルを再読み込みしてみましょう。TF SavedModel 形式としてモデルを保存したため、`tf.saved_model.load()` を使用して再読み込みします。

In [ ]:
reloaded_model = tf.saved_model.load(SAVEDMODEL_PATH)

ここでは、任意の文でモデルをテストできます。以下の examples 変数に追加するだけです。

In [ ]:
def print_my_examples(inputs, results):
    result_for_printing = [
        f"input: {inputs[i]:<30} : score: {results[i][0]:.6f}"
        for i in range(len(inputs))
    ]
    print(*result_for_printing, sep="\n")
    print()


examples = [
    "this is such an amazing movie!",  # これは先ほど試したのと同じ文です
    "The movie was great!",
    "The movie was meh.",
    "The movie was okish.",
    "The movie was terrible...",
]

reloaded_results = reloaded_model.serve(keras.ops.convert_to_tensor(examples))
original_results = classifier_model(keras.ops.convert_to_tensor(examples))

print("Results from the saved model:")
print_my_examples(examples, reloaded_results)
print("Results from the model in memory:")
print_my_examples(examples, original_results)

## （オプション）オンライン予測を取得するために Agent Platform にモデルをデプロイする

モデルを TensorFlow SavedModel 形式にエクスポートしました。モデルがこの形式になれば、ウェブアプリケーション、JavaScript、モバイルアプリケーションなどから、モデルを「サービング」する多くの方法があります。

次に、SavedModel コマンドラインインターフェースのコマンド saved_model_cli を使用して、保存したモデルのシグネチャを出力します。コマンドラインインターフェースおよびサポートされている show や run コマンドの詳細については、[こちらのドキュメント](https://www.tensorflow.org/guide/saved_model#overview_of_commands)を参照してください。

[TF Serving](https://www.tensorflow.org/tfx/guide/serving) でモデルを使用したい場合は、名前付きシグネチャのいずれかを介して SavedModel が呼び出されることに注意してください。Python では、次のようにテストできます。

In [ ]:
!saved_model_cli show \
    --tag_set serve \
    --signature_def serving_default \
    --dir {SAVEDMODEL_PATH}

!find {SAVEDMODEL_PATH}
os.environ['SAVEDMODEL_PATH'] = SAVEDMODEL_PATH

In [ ]:
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
PROJECT = !gcloud config get-value project 2>/dev/null
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"
MODEL_DISPLAYNAME = f"classification-bert-{TIMESTAMP}"

print(f"MODEL_DISPLAYNAME: {MODEL_DISPLAYNAME}")

# from https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/pre-built-containers
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"
)

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
# GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "\nHere are your current buckets:"
    gcloud storage ls
fi

In [ ]:
!gcloud storage cp -r $SAVEDMODEL_PATH gs://$BUCKET/$MODEL_DISPLAYNAME

モデルを Agent Platform のモデルレジストリにアップロードします。

In [ ]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAYNAME,
    artifact_uri=f"gs://{BUCKET}/{MODEL_DISPLAYNAME}",
    serving_container_image_uri=SERVING_CONTAINER_IMAGE_URI,
)

エンドポイントを作成し、作成されたエンドポイントにモデルをデプロイします。

In [ ]:
MACHINE_TYPE = "n1-standard-4"

endpoint = uploaded_model.deploy(
    machine_type=MACHINE_TYPE,
    accelerator_type=None,
    accelerator_count=None,
)

モデルがエンドポイントにアップロードされたら、エンドポイントにクエリを送信してモデルから予測を取得できます。クエリはインスタンスのリストである必要があります。モデルのシグネチャから、クエリのキーは `text` である必要があることがわかります。

In [ ]:
instances = [
    {"text": ["I loved the movie and highly recomment it"]},
    {"text": ["It was an okay movie in my opinion"]},
    {"text": ["I hated the movie"]},
]

In [ ]:
response = endpoint.predict(instances=instances)

In [ ]:
print(" prediction:", response.predictions)

## クリーンアップ
オンライン予測用にモデルをエンドポイントにデプロイする場合、最小の min-replica-count は 1 であり、ノード時間ごとに課金されます。そのため、不要な課金を減らすためにエンドポイントを削除しましょう。エンドポイントを削除する前に、まずアタッチされているすべてのモデルのデプロイを解除（undeploy）します。

In [ ]:
endpoint.undeploy_all()

...その後、エンドポイントを削除します。

In [ ]:
endpoint.delete()

# 継続的な学習

このラボでは、テキスト分類器の訓練に Small BERT を選択しました。他にも利用可能な事前学習済み BERT モデルがあり、ここで見つけることができます。これらのいくつかを試してみることを検討してください。ただし、ファインチューニングに選択するモデルが大きいほど、訓練にかかる時間が長くなることに留意してください。

以下のようなモデルがあります。
  - [BERT-Base](https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3)、[Uncased](https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3)、および元の BERT の著者によって公開された学習済み重みを持つ [7 つの追加モデル](https://tfhub.dev/google/collections/bert/1)。
  - [Small BERTs](https://tfhub.dev/google/collections/bert/1): 同じ一般的なアーキテクチャを持ちながら、Transformer ブロックの数が少ない、または小さいため、速度、サイズ、品質のトレードオフを検討できます。
  - [ALBERT](https://tfhub.dev/google/collections/albert/1): レイヤ間でパラメータを共有することにより、（計算時間ではなく）モデルサイズを削減する 4 つの異なるサイズの「A Lite BERT」。
  - [BERT Experts](https://tfhub.dev/google/collections/experts/bert/1): すべて BERT-base アーキテクチャを持ちながら、ターゲットタスクとより密接に一致させるために異なる事前学習ドメインの選択肢を提供する 8 つのモデル。
  - [Electra](https://tfhub.dev/google/collections/electra/1): BERT と同じアーキテクチャ（3 つの異なるサイズ）を持ちながら、敵対的生成ネットワーク（GAN）に似た設定で識別器（discriminator）として事前学習されたモデル。
  - Talking-Heads Attention と Gated GELU を備えた BERT [[base](https://tfhub.dev/tensorflow/talkheads_ggelu_bert_en_base/1), [large](https://tfhub.dev/tensorflow/talkheads_ggelu_bert_en_large/1)]: Transformer アーキテクチャのコアに対する 2 つの改良が加えられています。

TensorFlow Hub のモデルドキュメントには、研究論文への詳細と参照が含まれています。

上記で利用可能なモデルの他に、より大規模でさらに高い精度が得られるモデルの[複数のバージョン](https://tfhub.dev/google/collections/transformer_encoders_text/1)もありますが、単一の GPU でファインチューニングするには大きすぎます。

#### License

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License